# PySpark + Hive Program

This notebook keeps the same Google Drive setup, input path, and output location from your Hive Hadoop notebook.

The only change is that each result is saved as **one single Parquet file** instead of a Spark output folder with multiple `part-*` files.

In [17]:
# Install PySpark if needed
# !pip install pyspark

In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
import shutil
import glob

In [50]:
# Create Spark Session with Hive Support
spark = SparkSession.builder \
    .appName("PatentCitationHiveSingleParquet") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark Session Started")

Spark Session Started


In [51]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
# Same input path as your original Hive Hadoop notebook
input_path = "/content/drive/MyDrive/Colab Notebooks/UCI/datasets/cite75_99.txt"

print("Input path:", input_path)

Input path: /content/drive/MyDrive/Colab Notebooks/UCI/datasets/cite75_99.txt


In [53]:
# Read citation data
df = spark.read.option("header", "true").csv(input_path)

df.show(5)
df.printSchema()

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows
root
 |-- CITING: string (nullable = true)
 |-- CITED: string (nullable = true)



In [54]:
# Clean and transform data
citations = df.select(
    col("CITING").cast("int").alias("citing"),
    col("CITED").cast("int").alias("cited")
).dropna()

citations.show(5)

+-------+-------+
| citing|  cited|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows


In [55]:
# Save as Hive table
citations.write.mode("overwrite").saveAsTable("patent_citations")

spark.sql("SHOW TABLES").show()

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|  default|patent_citations|      false|
+---------+----------------+-----------+



## Query 1: Patents cited more than 10 times

In [56]:
result10 = spark.sql("""
SELECT cited,
       COUNT(*) AS citation_count
FROM patent_citations
GROUP BY cited
HAVING citation_count > 10
ORDER BY citation_count DESC
""")

result10.show()

+-------+--------------+
|  cited|citation_count|
+-------+--------------+
|4723129|           779|
|4463359|           716|
|4740796|           678|
|4345262|           658|
|4558333|           654|
|4313124|           633|
|4683195|           631|
|4459600|           613|
|4683202|           605|
|3953566|           411|
|4367924|           401|
|3702886|           382|
|4733665|           360|
|3845770|           339|
|5103459|           311|
|4535773|           305|
|4358535|           292|
|4655771|           289|
|4901307|           287|
|3849241|           286|
+-------+--------------+
only showing top 20 rows


## Query 2: Patents cited more than 20 times

In [57]:
result20 = spark.sql("""
SELECT cited,
       COUNT(*) AS citation_count
FROM patent_citations
GROUP BY cited
HAVING citation_count > 20
ORDER BY citation_count DESC
""")

result20.show()

+-------+--------------+
|  cited|citation_count|
+-------+--------------+
|4723129|           779|
|4463359|           716|
|4740796|           678|
|4345262|           658|
|4558333|           654|
|4313124|           633|
|4683195|           631|
|4459600|           613|
|4683202|           605|
|3953566|           411|
|4367924|           401|
|3702886|           382|
|4733665|           360|
|3845770|           339|
|5103459|           311|
|4535773|           305|
|4358535|           292|
|4655771|           289|
|4901307|           287|
|3849241|           286|
+-------+--------------+
only showing top 20 rows


## Query 3: Patents cited more than 100 times

In [58]:
result100 = spark.sql("""
SELECT cited,
       COUNT(*) AS citation_count
FROM patent_citations
GROUP BY cited
HAVING citation_count > 100
ORDER BY citation_count DESC
""")

result100.show()

+-------+--------------+
|  cited|citation_count|
+-------+--------------+
|4723129|           779|
|4463359|           716|
|4740796|           678|
|4345262|           658|
|4558333|           654|
|4313124|           633|
|4683195|           631|
|4459600|           613|
|4683202|           605|
|3953566|           411|
|4367924|           401|
|3702886|           382|
|4733665|           360|
|3845770|           339|
|5103459|           311|
|4535773|           305|
|4358535|           292|
|4655771|           289|
|4901307|           287|
|3849241|           286|
+-------+--------------+
only showing top 20 rows


## Save Results as Single Parquet Files

In [59]:
# Same output location as your original Hive Hadoop notebook
output_base = "/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output"

print("Output folder:", output_base)

Output folder: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output


In [60]:
def save_single_parquet(df, output_file):
    """
    Save a Spark DataFrame as one real .parquet file.

    Spark normally writes a folder containing part files.
    This function:
    1. coalesces the DataFrame into one partition,
    2. writes to a temporary folder,
    3. moves the single part-*.parquet file to the final .parquet path,
    4. deletes the temporary folder.
    """
    temp_dir = output_file + "_temp"

    # Remove old temp folder or old final file
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    if os.path.exists(output_file):
        os.remove(output_file)

    # Write one Spark part file
    df.coalesce(1).write.mode("overwrite").parquet(temp_dir)

    # Find generated Parquet part file
    part_files = glob.glob(os.path.join(temp_dir, "part-*.parquet"))

    if not part_files:
        raise FileNotFoundError("No part parquet file found.")

    # Move part file to final single parquet file
    shutil.move(part_files[0], output_file)

    # Clean temporary Spark folder
    shutil.rmtree(temp_dir)

    print(f"Saved single Parquet file: {output_file}")

In [61]:
# Keep the same output names, but save them as single .parquet files
save_single_parquet(result10, os.path.join(output_base, "results_hive_10.parquet"))
save_single_parquet(result20, os.path.join(output_base, "results_hive_20.parquet"))
save_single_parquet(result100, os.path.join(output_base, "results_hive_100.parquet"))

print("Results saved successfully.")

Saved single Parquet file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/results_hive_10.parquet
Saved single Parquet file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/results_hive_20.parquet
Saved single Parquet file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/results_hive_100.parquet
Results saved successfully.


In [31]:
# Confirm files were created
!ls -lh "/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/M4"/*.parquet

-rw------- 1 root root 5.5K May 13 11:27 '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/results_hive_100.parquet'
-rw------- 1 root root 1.5M May 13 11:26 '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/results_hive_10.parquet'
-rw------- 1 root root 386K May 13 11:27 '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/results_hive_20.parquet'


In [32]:
# Optional: read back one Parquet file to verify
check_df = spark.read.parquet(
    "/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/Output/M4/results_hive_10.parquet"
)

check_df.show(10)

+-------+--------------+
|  cited|citation_count|
+-------+--------------+
|4723129|           779|
|4463359|           716|
|4740796|           678|
|4345262|           658|
|4558333|           654|
|4313124|           633|
|4683195|           631|
|4459600|           613|
|4683202|           605|
|3953566|           411|
+-------+--------------+
only showing top 10 rows


In [ ]:
## convert to csv file

In [63]:
import os
import shutil
import glob

def save_as_single_csv(df, output_file):
    temp_folder = output_file + "_temp"

    if os.path.exists(temp_folder):
        shutil.rmtree(temp_folder)

    if os.path.exists(output_file):
        os.remove(output_file)

    df.coalesce(1).write.mode("overwrite") \
        .option("header", True) \
        .csv(temp_folder)

    csv_file = glob.glob(temp_folder + "/part-*.csv")[0]

    shutil.move(csv_file, output_file)
    shutil.rmtree(temp_folder)

    print("Saved CSV file:", output_file)

In [65]:
import os
import shutil

output_path = "/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/M4"

# Ensure the output directory for M4 exists
os.makedirs(output_path, exist_ok=True)

# Manually clean up any previous Spark-generated directories or files
# before calling save_as_single_csv to prevent IsADirectoryError.
files_to_clean = [
    output_path + "/results_hive_10.csv",
    output_path + "/results_hive_20.csv",
    output_path + "/results_hive_100.csv"
]

for f_path in files_to_clean:
    if os.path.exists(f_path):
        if os.path.isdir(f_path):
            shutil.rmtree(f_path)
        else:
            os.remove(f_path)

save_as_single_csv(result10, files_to_clean[0])
save_as_single_csv(result20, files_to_clean[1])
save_as_single_csv(result100, files_to_clean[2])

Saved CSV file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/M4/results_hive_10.csv
Saved CSV file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/M4/results_hive_20.csv
Saved CSV file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Output/M4/results_hive_100.csv


In [46]:
# Stop Spark session
spark.stop()